# Chemprop v1 User Guide

**Version:** 1.0.0
**Created:** 2026-04-17  
**Last Updated:** 2026-04-21  
**Tested on:** 2026-04-21

**Audience:** Scientists and HPC users training molecular property prediction models with command-line workflows

## TL;DR

Chemprop is a message passing neural network toolkit for molecular property prediction. In this HPC environment, you use it as a command-line application via modules.

Quick start:

```bash
module load chemprop/1.7.1
chemprop_train --version
chemprop_train --help
```

This notebook focuses on reproducible CLI usage, SLURM-ready workflows, and troubleshooting on shared systems.

## Table of Contents

- [TL;DR](#tldr)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute QuickStart](#tldr-5-minute-quickstart)
  - [Step 1: Load the Module](#step-1-load-the-module)
  - [Step 2: Verify Setup](#step-2-verify-setup)
  - [Step 3: Run Your First Job](#step-3-run-your-first-job)
- [PART II — How to Use Chemprop (Read Carefully Once)](#part-ii--how-to-use-chemprop-read-carefully-once)
  - [What You Need to Know](#what-you-need-to-know)
  - [CPU vs GPU: When to Use Each](#cpu-vs-gpu-when-to-use-each)
  - [Keyboard Shortcuts / Options](#keyboard-shortcuts--options)
  - [Resource Requests](#resource-requests)
- [PART III — Workflows (Use As Needed)](#part-iii--workflows-use-as-needed)
  - [Train a Regression Model](#train-a-regression-model)
    - [Predict from a Saved Checkpoint](#predict-from-a-saved-checkpoint)
    - [Generate Molecular Fingerprints](#generate-molecular-fingerprints)
  - [Binary Classification with Scaffold-Balanced Split](#binary-classification-with-scaffold-balanced-split)
  - [Multi-Task Regression](#multi-task-regression)
  - [Hyperparameter Optimization and Best-Config Training](#hyperparameter-optimization-and-best-config-training)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Symptom-Based Lookup](#symptom-based-lookup)
  - [FAQ](#faq)
  - [External Resources](#external-resources)

## PART I — Getting Started (Read Once)

### TL;DR: 5-Minute QuickStart

```bash
module load chemprop/1.7.1
chemprop_train --version
chemprop_train --help
```

### Step 1: Load the Module

In [ ]:
%%bash
module avail chemprop 2>&1 | head -40

**Expected output:**
```text
One or more Chemprop modules are listed (for example chemprop/1.7.1).
```

In [ ]:
%%bash
module load chemprop/1.7.1
module list 2>&1 | head -40

**Expected output:**
```text
The loaded module list includes chemprop/1.7.1.
```

### Step 2: Verify Setup

In [ ]:
%%bash
module load chemprop/1.7.1
chemprop_train --help 2>&1 | head -30

**Expected output:**
```text
Usage text for chemprop_train appears with available options.
```

Success! The command-line interface is available in your environment.

### Step 3: Run Your First Job

In [ ]:
%%bash
module load chemprop/1.7.1
for cmd in chemprop_train chemprop_predict chemprop_hyperopt chemprop_interpret chemprop_fingerprint chemprop_web; do
  echo "===== ${cmd} --help ====="
  ${cmd} --help > ${cmd}_help.txt 2>&1 || true
  head -20 ${cmd}_help.txt
  echo
done

**Expected output:**
```text
Help summaries are printed for each Chemprop CLI command, and full help text is saved to *_help.txt files.
```

Your first job is complete: you verified all major Chemprop entry points and captured full option lists.

## PART II — How to Use Chemprop (Read Carefully Once)

### What You Need to Know

- Chemprop accepts molecular data in CSV format (SMILES plus targets).
- Common task types: regression, classification, multiclass, spectra.
- Primary workflow is train first, then predict using saved checkpoints.
- In this environment, use module commands for setup and CLI commands for execution.

| Platform | Status | Notes |
|---|---|---|
| CPU-only nodes | Supported | Good for small datasets and debugging |
| GPU nodes | Supported | Recommended for large datasets and ensembles |
| Login node | Limited use | Use for setup, help, and lightweight checks only |

### CPU vs GPU: When to Use Each

Use CPU when:
- you are validating data format
- you are testing command syntax
- your dataset is small

Use GPU when:
- training deep ensembles
- using large datasets
- time-to-result matters

Decision guide:

```text
Prototype or small data?
  yes -> CPU
  no  -> GPU
```

### Options

Chemprop is non-interactive CLI software, so focus on options instead of keyboard shortcuts.

Key command families:
- `chemprop_train`: model training and cross-validation
- `chemprop_predict`: batch inference with saved checkpoints
- `chemprop_hyperopt`: hyperparameter optimization
- `chemprop_interpret`: model interpretation tools
- `chemprop_fingerprint`: latent/fingerprint extraction

To inspect full flags for any command:

```bash
module load chemprop/1.7.1
chemprop_train --help
```

### Resource Requests

Chemprop training is compute-intensive. Use scheduler allocations for real training jobs.

Example SLURM template:

```bash
#!/bin/bash
#SBATCH --job-name=chemprop-train
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=04:00:00

module load chemprop/1.7.1
chemprop_train --data_path data.csv --dataset_type regression --save_dir checkpoints
```

For help-only and metadata checks, no allocation is needed on most clusters.

## PART III — Workflows (Use As Needed)

**How these workflows are organized:**

- `###` sections are standalone — each uses its own working directory and can be run in any order.
- `####` sections are sequential — they depend on the parent section's output and share the `./workflow` directory. Run the parent first.
- File-existence checks are performed before each job submission and exit with a clear message if prerequisites are missing.

### Train a Regression Model

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow
rm -rf "${WORKDIR}"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow1.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf1
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow1.out

set -euo pipefail

module load chemprop/1.7.1

echo "Creating synthetic regression dataset..."
cat > regression_train.csv << 'CSV'
smiles,target
CC,0.10
CCC,0.20
CCCC,0.30
CCCCC,0.40
CCCCCC,0.50
C=O,0.60
CC=O,0.70
CCC=O,0.80
CCO,0.90
CCCO,1.00
CCCCO,1.10
CCN,1.20
CCCN,1.30
CCCCN,1.40
c1ccccc1,1.50
c1ccncc1,1.60
c1ccoc1,1.70
CCCl,1.80
CCCCl,1.90
CCBr,2.00
CCCBr,2.10
COC,2.20
CCOC,2.30
CCCOC,2.40
CC(=O)O,2.50
CC(=O)N,2.60
CCS,2.70
CCCS,2.80
CCF,2.90
CCCF,3.00
CSV

echo "Training chemprop model..."
chemprop_train \
  --data_path regression_train.csv \
  --dataset_type regression \
  --save_dir checkpoints_reg \
  --epochs 3 \
  --split_sizes 0.8 0.1 0.1 \
  --num_workers 0 \
  > train.log 2>&1

echo "Verifying concrete outputs..."
test -f checkpoints_reg/fold_0/model_0/model.pt
test -f checkpoints_reg/fold_0/test_smiles.csv

find checkpoints_reg -maxdepth 4 -type f | sort > output_manifest.txt

echo "Workflow 1 completed successfully."
echo "Artifacts:"
echo "- regression_train.csv"
echo "- train.log"
echo "- checkpoints_reg/fold_0/model_0/model.pt"
echo "- checkpoints_reg/fold_0/test_smiles.csv"
echo "- output_manifest.txt"
SBATCH

sbatch chemprop_workflow1.sbatch

**Expected output:**
```text
The command runs successfully and Chemprop creates training artifacts inside ./workflow.

Chemprop-generated outputs:
- checkpoints_reg/fold_0/model_0/model.pt (binary; listing only)
- checkpoints_reg/fold_0/test_smiles.csv

Readable file preview (representative lines):
- checkpoints_reg/fold_0/test_smiles.csv:
  - smiles
  - CCO
  - CCN

Note: exact test-set rows can vary with data splitting.
```

#### Predict from a Saved Checkpoint

> **Prerequisites:** Complete [Train a Regression Model](#train-a-regression-model) first.

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow

echo "Checking Workflow 1 prerequisites..."
test -f "${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt" || {
  echo "Missing prerequisite: ${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt"
  echo "Run Workflow 1 first."
  exit 1
}
test -f "${WORKDIR}/checkpoints_reg/fold_0/test_smiles.csv" || {
  echo "Missing prerequisite: ${WORKDIR}/checkpoints_reg/fold_0/test_smiles.csv"
  echo "Run Workflow 1 first."
  exit 1
}

mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow2.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf2
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow2.out

set -euo pipefail

module load chemprop/1.7.1

test -f checkpoints_reg/fold_0/model_0/model.pt
test -f checkpoints_reg/fold_0/test_smiles.csv

echo "Running chemprop_predict using Workflow 1 artifacts..."
chemprop_predict \
  --test_path checkpoints_reg/fold_0/test_smiles.csv \
  --preds_path preds_reg.csv \
  --checkpoint_dir checkpoints_reg \
  > predict.log 2>&1

test -f preds_reg.csv
test -s preds_reg.csv

echo "Prediction file preview:"
head -n 10 preds_reg.csv

echo "Workflow 2 completed successfully."
SBATCH

sbatch chemprop_workflow2.sbatch

**Expected output:**
```text
The command checks for Workflow 1 artifacts first and exits with a clear message if they are missing.
If prerequisites exist, it runs successfully and Chemprop writes predictions.

Chemprop-generated outputs:
- preds_reg.csv

Readable file preview (representative lines):
- preds_reg.csv:
  - smiles,prediction
  - CCO,0.91
  - CCN,1.18

Note: prediction values vary by run and model state.
```

#### Generate Molecular Fingerprints

> **Prerequisites:** Complete [Train a Regression Model](#train-a-regression-model) first.

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow

echo "Checking Workflow 1 prerequisites..."
test -f "${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt" || {
  echo "Missing prerequisite: ${WORKDIR}/checkpoints_reg/fold_0/model_0/model.pt"
  echo "Run Workflow 1 first."
  exit 1
}
test -f "${WORKDIR}/regression_train.csv" || {
  echo "Missing prerequisite: ${WORKDIR}/regression_train.csv"
  echo "Run Workflow 1 first."
  exit 1
}

mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow3.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf3
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow3.out

set -euo pipefail

module load chemprop/1.7.1

test -f checkpoints_reg/fold_0/model_0/model.pt
test -f regression_train.csv

echo "Running chemprop_fingerprint using Workflow 1 artifacts..."
chemprop_fingerprint \
  --test_path regression_train.csv \
  --preds_path fingerprint.csv \
  --checkpoint_dir checkpoints_reg \
  --fingerprint_type MPN \
  > fingerprint.log 2>&1

test -f fingerprint.csv
test -s fingerprint.csv

echo "Fingerprint file preview:"
head -n 5 fingerprint.csv

echo "Workflow 3 completed successfully."
SBATCH

sbatch chemprop_workflow3.sbatch

**Expected output:**
```text
The command checks for Workflow 1 artifacts first and exits with a clear message if they are missing.
If prerequisites exist, it runs successfully and Chemprop writes fingerprints.

Chemprop-generated outputs:
- fingerprint.csv

Readable file preview (representative lines):
- fingerprint.csv:
  - smiles,fp_0,fp_1,fp_2,...
  - CC,0.12,-0.03,0.77,...
  - CCC,0.18,-0.10,0.81,...

Note: fingerprint dimensionality and numeric values depend on model configuration.
```

### Binary Classification with Scaffold-Balanced Split

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow_cls
rm -rf "${WORKDIR}"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow4.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf4
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow4.out

set -euo pipefail

module load chemprop/1.7.1

echo "Creating synthetic binary classification dataset..."
cat > classification_train.csv << 'CSV'
smiles,active
Cc1ccccc1,0
Nc1ccccc1,1
Oc1ccccc1,0
Clc1ccccc1,1
Fc1ccccc1,0
Brc1ccccc1,1
COc1ccccc1,0
N#Cc1ccccc1,1
CC(=O)c1ccccc1,0
c1ccc2ccccc2c1,1
c1ccncc1,0
Cc1ccccn1,1
Cc1ccncc1,0
c1ccoc1,1
c1ccsc1,0
CC(=O)Oc1ccccc1C(=O)O,1
CC(=O)Nc1ccc(O)cc1,0
CC(C)Cc1ccc(C(C)C(=O)O)cc1,1
NC(=O)c1ccccc1,0
COC(=O)c1ccccc1,1
c1ccc(Oc2ccccc2)cc1,0
C=Cc1ccccc1,1
c1cnccn1,0
Cc1ccc(C(=O)O)cc1,1
CC(C)(C)c1ccccc1,0
c1ccc2[nH]ccc2c1,1
c1ccc2[nH]cnc2c1,0
OC(=O)Cc1ccccc1,1
c1ccc(-c2ccccc2)cc1,0
CC1=CC(=O)c2ccccc2C1=O,1
CSV

echo "Training binary classification model with scaffold-balanced split..."
chemprop_train \
  --data_path classification_train.csv \
  --dataset_type classification \
  --split_type scaffold_balanced \
  --metric auc \
  --save_dir checkpoints_cls \
  --epochs 5 \
  --num_workers 0 \
  > train_cls.log 2>&1

test -f checkpoints_cls/fold_0/model_0/model.pt
test -f checkpoints_cls/fold_0/test_smiles.csv

find checkpoints_cls -maxdepth 4 -type f | sort > output_manifest_cls.txt

echo "Workflow 4 completed successfully."
echo "Artifacts:"
echo "- classification_train.csv"
echo "- train_cls.log"
echo "- checkpoints_cls/fold_0/model_0/model.pt"
echo "- checkpoints_cls/fold_0/test_smiles.csv"
SBATCH

sbatch chemprop_workflow4.sbatch

**Expected output:**
```text
The workflow trains a binary classification model on 30 molecules with chemically diverse Bemis-Murcko scaffolds, ensuring the test set contains structurally distinct compounds.

Chemprop-generated outputs:
- checkpoints_cls/fold_0/model_0/model.pt (classification model checkpoint)
- checkpoints_cls/fold_0/test_smiles.csv (scaffold-split test SMILES)
- train_cls.log

Readable file preview (representative lines of train_cls.log):
- Epoch 0 train auc = 0.62, val auc = 0.55
- Epoch 4 train auc = 0.79, val auc = 0.61

Note: scaffold-balanced splitting places structurally distinct Bemis-Murcko scaffold clusters in separate train, validation, and test sets, providing a more realistic generalization estimate than random splitting. AUC values on this 30-molecule toy dataset will be variable; production virtual screening datasets with 500+ compounds yield stable metrics.
```

### Multi-Task Regression

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow_mt
rm -rf "${WORKDIR}"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow5.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf5
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow5.out

set -euo pipefail

module load chemprop/1.7.1

echo "Creating synthetic multi-task regression dataset..."
cat > multitask_train.csv << 'CSV'
smiles,logD,solubility,herg_pic50
CC,0.50,2.10,4.20
CCC,0.80,1.90,4.50
CCCC,1.20,1.60,4.80
CCCCC,1.60,1.30,5.10
CCCCCC,2.00,1.00,5.40
C=O,-1.30,3.90,3.80
CC=O,-0.80,3.50,4.00
CCC=O,-0.30,3.10,4.30
CCO,-0.30,3.80,4.10
CCCO,0.20,3.40,4.40
CCCCO,0.70,3.00,4.70
CCN,-0.50,3.20,5.20
CCCN,0.00,2.80,5.50
CCCCN,0.50,2.40,5.80
c1ccccc1,2.10,1.80,5.30
c1ccncc1,1.00,2.60,5.70
c1ccoc1,1.30,2.20,5.40
CCCl,1.40,1.70,4.60
CCCCl,1.80,1.40,4.90
CCBr,1.60,1.50,4.70
CCCBr,2.00,1.20,5.00
COC,-0.10,3.30,4.20
CCOC,0.40,3.00,4.50
CCCOC,0.90,2.60,4.80
CC(=O)O,-0.30,3.70,4.10
CC(=O)N,-1.00,3.20,4.60
CCS,0.90,2.00,4.80
CCCS,1.30,1.70,5.10
CCF,0.20,2.80,4.30
CCCF,0.60,2.50,4.60
CSV

echo "Training multi-task regression model (3 targets: logD, solubility, herg_pic50)..."
chemprop_train \
  --data_path multitask_train.csv \
  --dataset_type regression \
  --save_dir checkpoints_mt \
  --epochs 5 \
  --num_workers 0 \
  > train_mt.log 2>&1

test -f checkpoints_mt/fold_0/model_0/model.pt

find checkpoints_mt -maxdepth 4 -type f | sort > output_manifest_mt.txt

echo "Workflow 5 completed successfully."
echo "Artifacts:"
echo "- multitask_train.csv"
echo "- train_mt.log (per-task RMSE for logD, solubility, herg_pic50)"
echo "- checkpoints_mt/fold_0/model_0/model.pt"
SBATCH

sbatch chemprop_workflow5.sbatch

**Expected output:**
```text
The workflow trains a single model with three output heads simultaneously predicting logD, solubility, and hERG pIC50 for 30 molecules. Chemprop auto-detects the task count from the number of numeric columns in the CSV header.

Chemprop-generated outputs:
- checkpoints_mt/fold_0/model_0/model.pt (multi-task model checkpoint)
- train_mt.log (training log with per-task metrics)

Readable file preview (representative lines of train_mt.log):
- Task 0 (logD)       val rmse = 0.41
- Task 1 (solubility) val rmse = 0.38
- Task 2 (herg_pic50) val rmse = 0.35

Note: multi-task training shares the message passing encoder across all tasks. A single model learns a joint molecular representation, which often improves performance on correlated endpoints compared to independent single-task models. To predict with this model, use chemprop_predict --checkpoint_dir checkpoints_mt; the output will contain one prediction column per task.
```

### Hyperparameter Optimization and Best-Config Training

In [ ]:
%%bash
set -euo pipefail

module load chemprop/1.7.1

WORKDIR=./workflow_hyperopt
rm -rf "${WORKDIR}"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow6.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-wf6
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow6.out

set -euo pipefail

module load chemprop/1.7.1

echo "Creating synthetic regression dataset for hyperopt campaign..."
cat > hyperopt_train.csv << 'CSV'
smiles,target
CC,0.10
CCC,0.20
CCCC,0.30
CCCCC,0.40
CCCCCC,0.50
C=O,0.60
CC=O,0.70
CCC=O,0.80
CCO,0.90
CCCO,1.00
CCCCO,1.10
CCN,1.20
CCCN,1.30
CCCCN,1.40
c1ccccc1,1.50
c1ccncc1,1.60
c1ccoc1,1.70
CCCl,1.80
CCCCl,1.90
CCBr,2.00
CCCBr,2.10
COC,2.20
CCOC,2.30
CCCOC,2.40
CC(=O)O,2.50
CC(=O)N,2.60
CCS,2.70
CCCS,2.80
CCF,2.90
CCCF,3.00
CSV

echo "Step 1: Running hyperparameter search (3 trials, 3 epochs each)..."
chemprop_hyperopt \
  --data_path hyperopt_train.csv \
  --dataset_type regression \
  --num_iters 3 \
  --epochs 3 \
  --config_save_path best_config.json \
  --save_dir hyperopt_out \
  > hyperopt.log 2>&1

test -f best_config.json

echo "Best hyperparameter config:"
cat best_config.json

echo "Step 2: Training final model with best discovered configuration..."
chemprop_train \
  --config_path best_config.json \
  --data_path hyperopt_train.csv \
  --dataset_type regression \
  --save_dir checkpoints_best \
  --epochs 5 \
  --num_workers 0 \
  > train_best.log 2>&1

test -f checkpoints_best/fold_0/model_0/model.pt

echo "Workflow 6 completed successfully."
echo "Artifacts:"
echo "- best_config.json (optimized hyperparameter configuration)"
echo "- hyperopt.log"
echo "- checkpoints_best/fold_0/model_0/model.pt (final model)"
echo "- train_best.log"
SBATCH

sbatch chemprop_workflow6.sbatch

**Expected output:**
```text
The workflow runs three hyperparameter search trials with chemprop_hyperopt, then retrains a final model using the best discovered configuration.

Chemprop-generated outputs:
- best_config.json (best hyperparameter configuration)
- hyperopt_out/ (trial checkpoints and logs)
- checkpoints_best/fold_0/model_0/model.pt (final optimized model)
- hyperopt.log, train_best.log

Readable file preview (representative best_config.json):
{
  "hidden_size": 300,
  "depth": 3,
  "dropout": 0.0,
  "ffn_num_layers": 2
}

Note: 3 trials demonstrates the workflow mechanics. For production use, run 20-50 trials with 20-30 epochs each (--num_iters 30 --epochs 30) to find meaningful architecture improvements. The saved best_config.json is reusable: supply it to chemprop_train --config_path on any compatible dataset to reproduce the optimized architecture without re-running the search.
```

## PART IV — Troubleshooting (Skim When Broken)

### Symptom-Based Lookup

| Symptom | Likely Cause | Fix |
|---|---|---|
| `chemprop_train: command not found` | Module not loaded | `module load chemprop/1.7.1` |
| `No module named chemprop` | Wrong Python environment | Reload module and open a new shell/notebook kernel |
| CUDA not detected | Running on CPU node or missing GPU allocation | Submit to GPU partition and request `--gres=gpu:1` |
| Out-of-memory during training | Batch/model too large | Reduce batch size, request more memory, or use gradient checkpointing |
| CSV parsing error | Invalid delimiter/header/SMILES field | Validate CSV header and data format against Chemprop docs |

### FAQ

**Q: Can I run Chemprop on a login node?**

A: Use login nodes for setup and help commands only. Use scheduler allocations for training and large predictions.

**Q: Which command should I start with?**

A: Start with `chemprop_train --help`, then run a short training job on a small dataset.

**Q: Where can I find end-to-end examples?**

A: See the local workshop and benchmark resources listed in External Resources below.

### External Resources

- Official repository: https://github.com/chemprop/chemprop
- Main publication: https://pubs.acs.org/doi/10.1021/acs.jcim.3c01250
- Documentation homepage: https://chemprop.readthedocs.io/